Тема 3. Применение условных вероятностей в анализе данных

На некотором наборе данных (выбрать самостоятельно, можно: https://www.kaggle.com/datasets) продемонстрировать применение условных вероятностей (см. пример с рейтингом автомобилей по типу кузова, Notebook с примером "Титаника"). Рассматриваемый набор данных необходимо описать (назначение, описание признаков).


Описание набора данных

**Iris** содержит **150 наблюдений** (по 50 для каждого из трёх видов). Каждый цветок описан четырьмя числовыми признаками в сантиметрах:

| Колонка | Описание |
|---|---|
| `sepal_length` | Длина чашелистика, см |
| `sepal_width` | Ширина чашелистика, см |
| `petal_length` | Длина лепестка, см |
| `petal_width` | Ширина лепестка, см |
| `species` | Вид ириса (`setosa`, `versicolor`, `virginica`) |
| `type` | Категория ширины лепестка: `narrow` (≤ 1,5 см) или `wide` (> 1,5 см) |
| `count` | Вспомогательная колонка из единиц для подсчёта наблюдений в сводной таблице |

**Пусть:**

Добавим бинарный признак `petal_group`: **wide**, если ширина лепестка > 1,5 см, иначе **narrow**. 

Источник: [UCI Iris Dataset](https://archive.ics.uci.edu/dataset/53/iris). 

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris()

# Даём столбцам короткие и понятные имена
columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
data = pd.DataFrame(iris.data, columns=columns)
data['species'] = [iris.target_names[i] for i in iris.target]

# Новый категориальный признак для расчёта условной вероятности
data['type'] = np.where(data['petal_width'] > 1.5, 'wide', 'narrow')
data['count'] = 1

data.head(10)

,sepal_length,sepal_width,petal_length,petal_width,species,type,count
0,5.1,3.5,1.4,0.2,setosa,narrow,1
1,4.9,3.0,1.4,0.2,setosa,narrow,1
2,4.7,3.2,1.3,0.2,setosa,narrow,1
3,4.6,3.1,1.5,0.2,setosa,narrow,1
4,5.0,3.6,1.4,0.2,setosa,narrow,1
5,5.4,3.9,1.7,0.4,setosa,narrow,1
6,4.6,3.4,1.4,0.3,setosa,narrow,1
7,5.0,3.4,1.5,0.2,setosa,narrow,1
8,4.4,2.9,1.4,0.2,setosa,narrow,1
9,4.9,3.1,1.5,0.1,setosa,narrow,1


In [6]:
print('Названия колонок:')
print(data.columns.tolist())
print()
print('Уникальные значения species:')
print(data['species'].unique())
print()
print('Уникальные значения type:')
print(data['type'].unique())

Названия колонок:
['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species', 'type', 'count']

Уникальные значения species:
<ArrowStringArray>
['setosa', 'versicolor', 'virginica']
Length: 3, dtype: str

Уникальные значения type:
<ArrowStringArray>
['narrow', 'wide']
Length: 2, dtype: str


In [7]:
species_probs = data.groupby('species').size().div(len(data))
print('P(species):')
print(species_probs)

P(species):
species
setosa        0.333333
versicolor    0.333333
virginica     0.333333
dtype: float64


In [8]:
pivot = pd.pivot_table(
    data,
    values='count',
    index=['species'],
    columns=['type'],
    aggfunc=np.sum,
    fill_value=0
)
pivot

type,narrow,wide
species,,
setosa,50,0
versicolor,45,5
virginica,3,47


In [9]:
type_given_species = data.groupby(['species', 'type']).size().div(len(data)).div(
    species_probs, axis=0, level='species'
)
print('P(type | species):')
print(type_given_species)

P(type | species):
species     type  
setosa      narrow    1.00
versicolor  narrow    0.90
            wide      0.10
virginica   narrow    0.06
            wide      0.94
dtype: float64


In [10]:
type_probs = data.groupby('type').size().div(len(data))
print('P(type):')
print(type_probs)

P(type):
type
narrow    0.653333
wide      0.346667
dtype: float64


In [11]:
species_given_type = data.groupby(['type', 'species']).size().div(len(data)).div(
    type_probs, axis=0, level='type'
)
print('P(species | type):')
print(species_given_type)

P(species | type):
type    species   
narrow  setosa        0.510204
        versicolor    0.459184
        virginica     0.030612
wide    versicolor    0.096154
        virginica     0.903846
dtype: float64
